[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_26_mha_pure.ipynb)

# 🟡 Medium: Multi-Head Attention without Flax

*Attention & Transformers*
Problem 06's multi-head attention with no Flax — and short enough to write
from memory under interview conditions.

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model, num_heads, *, key): ...
    def __call__(self, Q, K, V): ...     # (B, seq, d_model) x3 -> (B, seq, d_model)
```

Four projections named `W_q`, `W_k`, `W_v`, `W_o`, each `Linear(d_model,
d_model)`, built from **four independent keys**:

```python
kq, kk, kv, ko = jax.random.split(key, 4)
```

One key used four times gives four identical matrices and raises nothing.

Scale by `1/sqrt(d_k)` where `d_k = d_model // num_heads` — not
`1/sqrt(d_model)`.

### Splitting and merging heads
The one place this reliably goes wrong. `reshape` re-divides memory, it never
reorders it, so `H` and `d_k` must be adjacent **and in that order**:

```python
split = lambda t: t.reshape(*t.shape[:-1], self.h, self.d_k).swapaxes(-3, -2)
...
o = o.swapaxes(-3, -2)                       # put H next to d_k again
o = o.reshape(*o.shape[:-2], self.h * self.d_k)   # only NOW may you collapse
```

Naming the batch (`B, S, D = Q.shape`) is fine here — the contract is 3-D and
the `nnx` original does exactly that. Negative axes (`*t.shape[:-1]`) are worth
the habit anyway, because they keep the same code working under `vmap`, but
nothing here requires it.

### A `Linear` is provided
The starter cell gives you this, the same way problem 06 gives you
`nnx.Linear`:

```python
class Linear:
    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias
```

### Why this exists alongside problem 06
Interview sandboxes often ship `jax` alone, so every `nnx.Module` problem here
is unrunnable there. The API is kept as close to the `nnx` version as it can
be — same class name, same constructor arguments, same `W_q`/`W_k`/`W_v`/`W_o`
attributes — so that practising this reinforces problem 06 rather than
competing with it. Only the key changes hands:

```python
MultiHeadAttention(8, 2, rngs=nnx.Rngs(params=0))   # nnx
MultiHeadAttention(8, 2, key=jax.random.key(0))     # here
```

`Linear` is handed to you in the starter cell for the same reason `nnx.Linear`
is: the exercise is the attention, not a dense layer typed four times. It is
six lines, and `b_23` is where you write it yourself.

A plain class is not a pytree, so `jax.grad(loss)(layer)` does not work —
differentiate with respect to the input instead.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, exactly as nnx.Linear is given to you in problem 06."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class MultiHeadAttention:
    """Q, K, V -> attention over num_heads heads."""

    def __init__(self, d_model, num_heads, *, key):
        pass  # Replace this

    def __call__(self, Q, K, V):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

mha = MultiHeadAttention(8, 2, key=jax.random.key(0))
print("W_q kernel:", mha.W_q.kernel.shape, " d_k:", mha.d_k)

x = jax.random.normal(jax.random.key(1), (2, 5, 8))
print("self-attention:", mha(x, x, x).shape)

xq = jax.random.normal(jax.random.key(2), (2, 3, 8))
print("different seq_q:", mha(xq, x, x).shape)

g = jax.grad(lambda v: jnp.sum(mha(v, v, v)))(x)
print("\nd/dx shape:", g.shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("mha_pure")

# hint("mha_pure")      # stuck? nudge without the answer
# solution("mha_pure")  # spoiler: the reference implementation
# status()              # your dashboard across all problems